# Gold Direction Predictor

Modelo de clasificación para predecir la dirección del precio del oro (XAU/USD).

## Objetivo
Predecir si el oro subirá o bajará al día siguiente usando variables macroeconómicas 
y técnicas como features.

## Features
- Retorno del DXY (índice del dólar)
- Cambio del VIX (índice de volatilidad)
- Retorno del S&P500
- Volatilidad histórica 7 días del oro
- Retorno del día anterior
- Distancia a MA7 y MA21

## Modelo
Random Forest Classifier con validación temporal out-of-sample

## Período
Entrenamiento: 2020-2024 | Test: 2024

    

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_auc_score)

In [5]:
# Descargar datos
print("Descargando datos...")
oro   = yf.download("GC=F",    start="2020-01-01", end="2024-12-31")
dxy   = yf.download("DX-Y.NYB", start="2020-01-01", end="2024-12-31")
vix   = yf.download("^VIX",    start="2020-01-01", end="2024-12-31")
sp500 = yf.download("^GSPC",   start="2020-01-01", end="2024-12-31")

# Construir features
df = pd.DataFrame()
df['oro_retorno']    = oro['Close'].pct_change() * 100
df['dxy_retorno']    = dxy['Close'].pct_change() * 100
df['vix_cambio']     = vix['Close'].pct_change() * 100
df['sp500_retorno']  = sp500['Close'].pct_change() * 100
df['volatilidad_7d'] = df['oro_retorno'].rolling(7).std()
df['retorno_ayer']   = df['oro_retorno'].shift(1)
df['ma7']            = oro['Close'].rolling(7).mean()
df['ma21']           = oro['Close'].rolling(21).mean()
df['dist_ma7']  = (oro['Close'].squeeze() - df['ma7']) / df['ma7'] * 100
df['dist_ma21'] = (oro['Close'].squeeze() - df['ma21']) / df['ma21'] * 100

# TARGET — lo que queremos predecir
# 1 = el oro subió mañana, 0 = bajó o se mantuvo
df['target'] = (df['oro_retorno'].shift(-1) > 0).astype(int)

# Limpiar
df = df.dropna()

print(f"Dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nDistribución del target:")
print(f"Sube (1): {df['target'].sum()} días ({df['target'].mean()*100:.1f}%)")
print(f"Baja (0): {(df['target']==0).sum()} días ({(1-df['target'].mean())*100:.1f}%)")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Descargando datos...
Dataset: 1237 filas, 11 columnas

Distribución del target:
Sube (1): 674 días (54.5%)
Baja (0): 563 días (45.5%)


In [6]:
# Split temporal
features = ['dxy_retorno', 'vix_cambio', 'sp500_retorno', 
            'volatilidad_7d', 'retorno_ayer', 'dist_ma7', 'dist_ma21']

X = df[features]
y = df['target']

split = int(len(df) * 0.8)
X_train = X.iloc[:split]
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

# Entrenar Random Forest Classifier
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
clf.fit(X_train, y_train)

# Predecir
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

# Evaluar
print("=== RESULTADOS DEL CLASIFICADOR ===")
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Baseline:  54.50% (predecir siempre 'sube')")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Baja', 'Sube'])}")

=== RESULTADOS DEL CLASIFICADOR ===

Accuracy:  50.40%
Baseline:  54.50% (predecir siempre 'sube')
AUC-ROC:   0.4843

              precision    recall  f1-score   support

        Baja       0.38      0.22      0.28       107
        Sube       0.55      0.72      0.62       141

    accuracy                           0.50       248
   macro avg       0.46      0.47      0.45       248
weighted avg       0.47      0.50      0.47       248

